# 1 · Ingestion: from corpus files to searchable chunks

This notebook calls the **real functions** of the ingestion scripts, one step at a time, on one document (the German
label v4.2), and shows what each step produces and why.

```
corpus files ─► Step 1  registry CSVs → BigQuery tables + views        src/01_load_tables.py
             ─► Step 2  Document AI Layout Parser → chunk rows          src/02_parse.py
             ─► Step 3  quality gate (pages, OCR, tables, faithfulness) src/02a_check_parse.py
             ─► Step 4  split tables · tag templates · collapse copies  src/03_embed_and_load.py
             ─► Step 5  registry fields · embedding text · embedding   src/03_embed_and_load.py
             ─► Step 6  MERGE into BigQuery `chunks` + table facts     src/03_embed_and_load.py
             ─► Step 7  verify-searchable probe                        src/03_embed_and_load.py
             ─► Step 8  vector index (only at scale)                   src/04_create_index.py
```
`src/run_ingestion.py` runs these steps in this order under one run ID and stops at the first failure.

**Safe to run.** With `LIVE = False` nothing leaves your laptop: it reads the corpus, the cached Layout Parser results
(`output/parsed/`) and the files the pipeline wrote (`output/`). No Document AI call, no embedding, no BigQuery write.
`LIVE = True` adds read-only BigQuery queries and one embedding call; this notebook never writes anything.

In [1]:
LIVE = False   # False: recorded responses and local files, no cost. True: real Google Cloud calls (small cost).

from nb_setup import setup, short, table, module, functions
SUBMISSION = setup(live=LIVE)

submission : /Users/vikassingh/Downloads/GIT/GCP_FDE/Session-08-UseCases-Assignments/submission
mode       : OFFLINE: recorded responses, no cost
project    : tredence-fde-1 · answers: gemini-3.5-flash-lite · agent: gemini-3.1-flash-lite · embeddings: gemini-embedding-001


## The services, and what you would use on Databricks

| Step | On GCP (this project) | On Databricks |
|---|---|---|
| Original files | Cloud Storage bucket (`raw/`) | Unity Catalog volume (or S3 / ADLS) |
| Registry tables + views | BigQuery tables + views | Delta tables + views in Unity Catalog |
| Parse PDF / DOCX / scans | Document AI **Layout Parser** (managed, OCR included) | `ai_parse_document` (AI Functions) or an open-source parser (Docling, Unstructured) in a job |
| Embeddings | Vertex AI `gemini-embedding-001` (768 numbers) | a Model Serving embedding endpoint (e.g. GTE / BGE), or an external-model endpoint that calls Vertex AI |
| Store + search | BigQuery `chunks` + hybrid `VECTOR_SEARCH` | Delta table + Mosaic AI Vector Search index (hybrid search, filters) |
| Pipeline run | `run_ingestion.py` (local); designed: Eventarc → Cloud Run | a Databricks Job (file-arrival trigger, Auto Loader) |
| Health records | BigQuery `pipeline_health` | a Delta table + Lakehouse Monitoring / SQL alerts |

The logic (filters, templates, copies, governance fields, checks) is plain Python and moves as is; only the service
calls change. Databricks product names are as of 2026: check the current docs before building.

In [2]:
import config
for name in ["PROJECT_ID", "GCS_BUCKET", "BQ_DATASET", "DOCUMENT_AI_LOCATION", "PROCESSOR_VERSION", "CHUNK_SIZE",
             "INCLUDE_ANCESTOR_HEADINGS", "EMBEDDING_MODEL", "EMBEDDING_DIMENSION", "EMBEDDING_LOCATION", "CORPUS_DIR"]:
    print(f"{name:26} {getattr(config, name)}")
print(f"{'SOURCE_FILES':26} {len(config.SOURCE_FILES)} files (the allowlist, read from the registry CSV)")

PROJECT_ID                 tredence-fde-1
GCS_BUCKET                 tredence-fde-1-medevidence-vks
BQ_DATASET                 medevidence_v1_vks
DOCUMENT_AI_LOCATION       us
PROCESSOR_VERSION          pretrained-layout-parser-v1.5-2025-08-25
CHUNK_SIZE                 512
INCLUDE_ANCESTOR_HEADINGS  True
EMBEDDING_MODEL            gemini-embedding-001
EMBEDDING_DIMENSION        768
EMBEDDING_LOCATION         us-central1
CORPUS_DIR                 /Users/vikassingh/Downloads/GIT/GCP_FDE/Session-08-UseCases-Assignments/Data-Corpus-v1/01-MedEvidence-Large-Corpus
SOURCE_FILES               8 files (the allowlist, read from the registry CSV)


## Step 0 · provision and preflight (run once, not repeated here)

- `src/00_provision.py` creates what the pipeline needs, and re-running it creates nothing new: the bucket (uploads
  only allowlisted files, skips a file whose MD5 already matches), the BigQuery dataset, and the Layout Parser processor
  (its ID is saved in `.state/`, so later scripts find it).
- `src/00_preflight.py` checks 8 things before any work: login, corpus, bucket, BigQuery, processor, and one live call
  each to embeddings, Gemini and the Ranking API (including that it ranks a renal passage above a storage passage).

In [3]:
from google.cloud import documentai

processor_id = config.PROCESSOR_ID_FILE.read_text().strip()
print("processor ID (from .state):", processor_id)
# The full name every Document AI call uses. It is pinned to one processor VERSION, so a Google update cannot
# silently change our chunks. (A static helper: no client, no call.)
print(documentai.DocumentProcessorServiceClient.processor_version_path(
    config.PROJECT_ID, config.DOCUMENT_AI_LOCATION, processor_id, config.PROCESSOR_VERSION))
print("one file in the bucket:", config.gcs_uri("DE_AX17_Current_Label_v4.2.pdf"))

processor ID (from .state): 8b959a2b083c4bfd
projects/tredence-fde-1/locations/us/processors/8b959a2b083c4bfd/processorVersions/pretrained-layout-parser-v1.5-2025-08-25
one file in the bucket: gs://tredence-fde-1-medevidence-vks/raw/DE_AX17_Current_Label_v4.2.pdf


## Step 1 · registry CSVs → BigQuery tables + views (`01_load_tables.py`)

The registry decides **which documents exist, and their status, country, version and dates**. It is loaded as plain
tables (never embedded), because those fields become SQL filters that run *before* any search.

- `read_documents()` cleans each row (`FINAL → APPROVED`, countries → a list, dates → UTC timestamps), and
  `_derive_valid_to()` gives each superseded label its end date and `superseded_by` (from the next version).
- Views add what changes with time: `v_documents.is_current` is computed at query time (`CURRENT_TIMESTAMP()`), so
  nothing goes stale; `v_adverse_events` keeps one row per study × event (the CSV repeats each row for 4 subgroups).

In [4]:
load = module("01_load_tables")
functions(load)

,line,function,what it does
0,58,read_effective_times,document_id → exact effective time (UTC) from ...
1,83,_utc_date,A date with no time means 00:00 UTC that day (...
2,91,_effective_from,"The registry date at 00:00 UTC (D3), or the ex..."
3,103,_clean_document,Normalize one registry or supplement row into ...
4,137,_derive_valid_to,Set `valid_to` and `superseded_by` from the ne...
5,160,read_documents,"Registry rows plus supplement rows, cleaned, w..."
6,170,read_adverse_events,AE rows with numeric rates; values are not cha...
7,190,check_quality,Fail the run before loading if the data breaks...
8,285,_to_json,BigQuery JSON loads take timestamps as ISO str...
9,290,load_snapshot,Replace a table with a full snapshot in one at...


In [5]:
documents = load.read_documents()
table(documents, ["document_id", "filename", "countries", "status", "raw_status", "version",
                  "effective_from", "valid_to", "superseded_by", "authority", "document_type"])

,document_id,filename,countries,status,raw_status,version,effective_from,valid_to,superseded_by,authority,document_type
0,DE-AX17-LBL-042,DE_AX17_Current_Label_v4.2.pdf,[Germany],APPROVED,APPROVED,4.2,2026-09-15 00:00:00+00:00,NaT,NaN,REGULATORY_LABEL,LABEL
1,DE-AX17-LBL-038,DE_AX17_Superseded_Label_v3.8.pdf,[Germany],SUPERSEDED,SUPERSEDED,3.8,2025-02-03 00:00:00+00:00,2026-09-15 00:00:00+00:00,DE-AX17-LBL-042,REGULATORY_LABEL,LABEL
2,US-AX17-PI-050,US_AX17_Prescribing_Information_v5.0.pdf,[United States],APPROVED,APPROVED,5.0,2026-08-01 00:00:00+00:00,NaT,NaN,REGULATORY_LABEL,LABEL
3,HN-204,HN-204_Clinical_Study.pdf,[GLOBAL],APPROVED,FINAL,1.0,2025-06-12 00:00:00+00:00,NaT,NaN,CLINICAL_STUDY,STUDY
4,HN-311,HN-311_Clinical_Study.pdf,[GLOBAL],APPROVED,FINAL,1.0,2026-01-18 00:00:00+00:00,NaT,NaN,CLINICAL_STUDY,STUDY
5,HN-402,HN-402_Clinical_Study.pdf,[GLOBAL],APPROVED,FINAL,1.0,2026-05-22 00:00:00+00:00,NaT,NaN,CLINICAL_STUDY,STUDY
6,SB-AX17-2026-09,SB_AX17_2026_09_Safety_Bulletin.docx,"[Germany, Singapore, UAE]",APPROVED,APPROVED,1.0,2026-10-07 09:00:00+00:00,NaT,NaN,SAFETY_BULLETIN,BULLETIN
7,DE-AX17-HSL-2019-2021,DE_AX17_Historical_Scanned_Safety_Letters_2019...,[Germany],SUPERSEDED,SUPERSEDED,archive,2019-01-01 00:00:00+00:00,2022-01-01 00:00:00+00:00,NaN,SAFETY_LETTER,LETTER


In [6]:
adverse_events = load.read_adverse_events()
print(len(adverse_events), "adverse-event rows ·",
      len({(r["study_id"], r["event"]) for r in adverse_events}), "distinct study × event (what v_adverse_events keeps)")
table(adverse_events[:6])

72 adverse-event rows · 18 distinct study × event (what v_adverse_events keeps)


,study_id,event,treatment_rate_pct,control_rate_pct,renal_subgroup,source_file
0,HN-204,nausea,12.7,6.0,normal,adverse_event_registry.csv
1,HN-204,nausea,12.7,6.0,mild,adverse_event_registry.csv
2,HN-204,nausea,12.7,6.0,moderate,adverse_event_registry.csv
3,HN-204,nausea,12.7,6.0,severe,adverse_event_registry.csv
4,HN-204,headache,14.0,6.6,normal,adverse_event_registry.csv
5,HN-204,headache,14.0,6.6,mild,adverse_event_registry.csv


In [7]:
from health import Health

health = Health("notebook")   # records in memory only: .save() is never called in this notebook
load.check_quality(documents, adverse_events, health)
table(health.rows, ["check", "status", "value", "threshold", "details"])

[check] 8 documents, 72 AE rows, 18 study+event pairs: all checks passed


,check,status,value,threshold,details
0,document_ids_unique,pass,8.0,no duplicates,None
1,documents_match_allowlist,pass,8.0,= 8 files,None
2,status_in_lifecycle,pass,0.0,= 0,None
3,countries_present,pass,0.0,= 0,None
4,superseded_consistency,pass,0.0,= 0,None
5,ae_complete_grid,pass,72.0,= studies × events × subgroups = 72,None
6,ae_rates_constant_across_subgroups,pass,18.0,1 rate pair per study+event,None
7,ae_studies_registered,pass,0.0,= 0,None


Every check becomes a row (in BigQuery `pipeline_health` when the script runs), so the health report shows trends,
not one printout. Explicit schemas mean BigQuery never guesses a column type:

In [8]:
for name, schema in [("documents", load.DOCUMENTS_SCHEMA), ("adverse_events", load.AE_SCHEMA)]:
    print(f"{name}:")
    for f in schema:
        print(f"   {f.name:22} {f.field_type:10} {f.mode:9} {f.description or ''}")

documents:
   document_id            STRING     REQUIRED  Controlled document ID, e.g. DE-AX17-LBL-042. Cite this.
   title                  STRING     NULLABLE  Readable title derived from the file name.
   filename               STRING     REQUIRED  Original file name in the corpus.
   product                STRING     REQUIRED  Product, e.g. AX-17. A hard filter: products are never mixed.
   countries              STRING     REPEATED  Jurisdictions the document applies to. GLOBAL = scientific evidence anywhere, regulatory guidance nowhere.
   status                 STRING     REQUIRED  Normalized lifecycle status used for filtering: DRAFT, REVIEW, APPROVED, SUPERSEDED, WITHDRAWN.
   raw_status             STRING     NULLABLE  Status exactly as in the source registry, e.g. FINAL (kept for audit).
   version                STRING     NULLABLE  Document version, e.g. 4.2.
   effective_from         TIMESTAMP  REQUIRED  When the document took effect (UTC).
   valid_to               TIMES

In [9]:
if LIVE:   # what BigQuery holds now (read-only)
    from google.cloud import bigquery
    bq = bigquery.Client(project=config.PROJECT_ID)
    rows = [dict(r.items()) for r in bq.query(f"""SELECT document_id, status, is_current, superseded_by
            FROM `{config.table_id('v_documents')}` ORDER BY document_id""").result()]
    display(table(rows))
else:
    print("Set LIVE = True to read v_documents from BigQuery (is_current is computed at query time).")

Set LIVE = True to read v_documents from BigQuery (is_current is computed at query time).


## Step 2 · parse with Document AI Layout Parser (`02_parse.py`)

Layout Parser reads each PDF, DOCX or scan and returns two trees: the **layout** (blocks: headings, paragraphs,
tables, images) and its own **chunks** (≤ 512 tokens, each starting with its heading path). The raw result is cached,
keyed by the file's MD5, so a re-run costs nothing and an edited file is parsed again automatically.

In [10]:
parse = module("02_parse")
functions(parse)

,line,function,what it does
0,71,_client,"Document AI is regional, so the client must us..."
1,78,_processor_version_name,"Full resource name of our processor, pinned to..."
2,84,_process_options,Layout Parser options: annotations plus its bu...
3,98,_parse_online,One synchronous call; the response comes back ...
4,109,_parse_batch,Asynchronous call for files the online call re...
5,159,_index_blocks,Map every layout block ID (including nested on...
6,183,_chunk_rows,Our chunk rows from Layout Parser's chunks.
7,218,_document_ids,"filename → document_id, from the BigQuery regi..."
8,226,parse_file,Return the raw Layout Parser result for one fi...
9,257,main,Parse every allowlisted file (or one) and writ...


In [11]:
FILE = "DE_AX17_Current_Label_v4.2.pdf"
DOC = next(d["document_id"] for d in documents if d["filename"] == FILE)
print("document:", DOC)
# The allowlist comes from the registry: only controlled documents are ever parsed (not the answer key).
for p in sorted(config.CORPUS_DIR.iterdir()):
    print("  ✓" if p.name in config.SOURCE_FILES else "  ✗", p.name)

document: DE-AX17-LBL-042
  ✗ CORPUS_SCALE_ESTIMATE.md
  ✓ DE_AX17_Current_Label_v4.2.pdf
  ✓ DE_AX17_Historical_Scanned_Safety_Letters_2019_2021.pdf
  ✓ DE_AX17_Superseded_Label_v3.8.pdf
  ✓ HN-204_Clinical_Study.pdf
  ✓ HN-311_Clinical_Study.pdf
  ✓ HN-402_Clinical_Study.pdf
  ✗ README.md
  ✓ SB_AX17_2026_09_Safety_Bulletin.docx
  ✗ TRAINER_GROUND_TRUTH.md
  ✓ US_AX17_Prescribing_Information_v5.0.pdf
  ✗ adverse_event_registry.csv
  ✗ document_registry.csv


In [12]:
import hashlib, json
path = config.CORPUS_DIR / FILE
md5 = hashlib.md5(path.read_bytes()).hexdigest()
cache = parse.PARSED_DIR / f"{path.stem}__cs{config.CHUNK_SIZE}__{md5[:12]}.json"
print("cache file:", cache.name, "· exists:", cache.exists())
assert cache.exists(), "No cached parse: parse_file() would call Document AI. Run src/02_parse.py first."
raw, mode = parse.parse_file(None, None, FILE, config.CHUNK_SIZE)   # a cache hit never uses the client
print("parse_file →", mode)
print("saved with the first parse:", json.loads(cache.read_text())["meta"])

cache file: DE_AX17_Current_Label_v4.2__cs512__a017cc207b07.json · exists: True
parse_file → cache
saved with the first parse: {'filename': 'DE_AX17_Current_Label_v4.2.pdf', 'md5': 'a017cc207b07fe22821135727917f065', 'mode': 'online', 'processor_version': 'pretrained-layout-parser-v1.5-2025-08-25', 'chunk_size': 512, 'include_ancestor_headings': True, 'table_annotation': True, 'image_annotation': True, 'parsed_at': '2026-10-15T11:42:38.100758+00:00'}


In [13]:
# The options _parse_online() sends: Layout Parser's own chunking + image and table descriptions.
print(parse._process_options(config.CHUNK_SIZE))
print("filled parts of the result:", {k: len(v) for k, v in raw.items() if v and hasattr(v, "__len__")})

layout_config {
  chunking_config {
    chunk_size: 512
    include_ancestor_headings: true
  }
  enable_image_annotation: true
  enable_table_annotation: true
}

filled parts of the result: {'documentLayout': 1, 'chunkedDocument': 1, 'blobAssets': 3}


In [14]:
from collections import Counter

kinds = {}
parse._index_blocks(raw["documentLayout"]["blocks"], kinds)   # block ID → kind, nested blocks included
print(len(raw["documentLayout"]["blocks"]), "top-level blocks →", len(kinds), "block IDs:", dict(Counter(kinds.values())))
chunks = raw["chunkedDocument"]["chunks"]
chart = next(c for c in chunks if "line graph" in c["content"])
print("\nraw chunk", chart["chunkId"], "· blocks", chart["sourceBlockIds"], "→", [kinds.get(b) for b in chart["sourceBlockIds"]])
print("BEFORE:", short(chart["content"], 240))
print("AFTER :", short(parse.ANNOTATION.sub("", chart["content"]).strip()))
print("moved to `annotations` (written by the parser's model, never cited as document text):")
print("       ", short(parse.ANNOTATION.findall(chart["content"])[0].strip(), 160))

13 top-level blocks → 48 block IDs: {'text/header': 4, 'text/paragraph': 16, 'text/footer': 16, 'image': 3, 'text/heading-2': 2, 'text/heading-1': 6, 'table': 1}

raw chunk c26 · blocks ['58', '59'] → ['text/heading-1', 'image']
BEFORE: # AX-17 Exposure by Renal Function ⏎  ⏎ __START_OF_ANNOTATION__This line graph, titled "AX-17 Exposure by Renal Function", illustrates the relationship between renal function (x-axis) and relative AUC (Area Under the Curve) (y-axis) for AX- …
AFTER : # AX-17 Exposure by Renal Function
moved to `annotations` (written by the parser's model, never cited as document text):
        This line graph, titled "AX-17 Exposure by Renal Function", illustrates the relationship between renal function (x-axis) and relative AUC (Area Under the Curve) …


In [15]:
rows = parse._chunk_rows(DOC, FILE, config.CHUNK_SIZE, raw)
print(len(chunks), "Layout Parser chunks →", len(rows), "rows for", DOC)
table([dict(r, annotations=len(r["annotations"])) for r in rows],
      ["chunk_id", "page_start", "page_end", "block_kinds", "annotations", "page_headers", "content"], width=60)

38 Layout Parser chunks → 38 rows for DE-AX17-LBL-042


,chunk_id,page_start,page_end,block_kinds,annotations,page_headers,content
0,DE-AX17-LBL-042-c0000,1,1,"[image, text/paragraph]",2,[AX-17 Germany - Approved Product Information ...,Document ID: DE-AX17-LBL-042 Status: APPROVED ...
1,DE-AX17-LBL-042-c0001,2,3,"[text/heading-2, text/paragraph]",0,[],## Indications and Population\n\nIndications a...
2,DE-AX17-LBL-042-c0002,2,2,"[text/heading-2, text/paragraph]",0,[],"## Indications and Population\n\nTables, warni..."
3,DE-AX17-LBL-042-c0003,2,2,"[text/heading-2, text/paragraph]",0,[],## Indications and Population\n\nHeadache was ...
4,DE-AX17-LBL-042-c0004,3,6,"[text/heading-2, text/paragraph]",0,[Renal Impairment],## Dosage and Administration\n\nIndications an...
5,DE-AX17-LBL-042-c0005,3,3,"[text/heading-2, text/paragraph]",0,[],"## Dosage and Administration\n\nTables, warnin..."
6,DE-AX17-LBL-042-c0006,3,3,"[text/heading-2, text/paragraph]",0,[],## Dosage and Administration\n\nDizziness was ...
7,DE-AX17-LBL-042-c0007,4,4,"[text/heading-2, text/paragraph]",0,[],## Dosage and Administration\n\nDosage and Adm...
8,DE-AX17-LBL-042-c0008,4,4,"[text/heading-2, text/paragraph]",0,[],"## Dosage and Administration\n\nTables, warnin..."
9,DE-AX17-LBL-042-c0009,5,5,"[text/heading-2, text/paragraph]",0,[Renal Impairment],## Dosage and Administration\n\nRenal Impairme...


### Explore the parse yourself
Change `page` (and `FILE` above) and read the chunks next to the PDF page itself; the second cell prints the layout
tree Layout Parser saw (indentation = nesting: a paragraph sits under its heading).

In [16]:
page = 6   # any page number
for r in rows:
    if r["page_start"] and r["page_start"] <= page <= (r["page_end"] or r["page_start"]):
        print(f"=== {r['chunk_id']} · pages {r['page_start']}-{r['page_end']} · {r['block_kinds']}")
        print(r["content"])
        for a in r["annotations"]:
            print("   [description written by the parser's model]", short(a, 300))

=== DE-AX17-LBL-042-c0004 · pages 3-6 · ['text/heading-2', 'text/paragraph']
## Dosage and Administration

Indications and Population. The controlled AX-17 evidence package for Germany included 48 evaluable participants in this analysis. Investigators recorded treatment-emergent findings, laboratory changes, dose interruptions, concomitant therapy, and clinically relevant renal and hepatic observations. Headache was reported in 14.3% of the described subgroup. This value is descriptive and must be interpreted only within the population, study, jurisdiction, version, and effective-date context in which it appears. Tables, warnings, captions, footnotes, and parent headings may modify the meaning of this paragraph and therefore form part of the retrievable evidence. HelixNova medical review requires source identity, document status, page context, and authority to remain traceable through downstream retrieval and generation. Indications and Population. The controlled AX-17 evidence package

In [17]:
def tree(blocks, depth=0, limit=[40]):
    for b in blocks:
        if limit[0] <= 0:
            return
        limit[0] -= 1
        kind = next(k for k in b if k.endswith("Block"))
        d = b[kind]
        text = {"textBlock": d.get("text", ""), "tableBlock": "<table>",
                "imageBlock": "<image> " + d.get("imageText", "")}.get(kind, kind)
        page_no = b.get("pageSpan", {}).get("pageStart")
        print("  " * depth + f"{b['blockId']:>3} {d.get('type', kind.removesuffix('Block')):10} p{page_no}: {text[:70]!r}")
        tree(d.get("blocks", []), depth + 1, limit)

tree(raw["documentLayout"]["blocks"])   # the first 40 blocks

  1 header     p1: 'AX-17 Germany - Approved Product Information v4.2'
  2 paragraph  p1: 'Document ID: DE-AX17-LBL-042 Status: APPROVED Effective: 15-Sep-2026 J'
  3 footer     p1: 'Synthetic training corpus - not for clinical use Page 1'
  4 image      p1: '<image> '
  5 image      p1: '<image> '
  6 heading-2  p2: 'Indications and Population'
    7 paragraph  p2: 'Indications and Population. The controlled AX-17 evidence package for '
    8 footer     p2: 'Synthetic training corpus - not for clinical use Page 2'
  9 heading-2  p3: 'Dosage and Administration'
   10 paragraph  p3: 'Indications and Population. The controlled AX-17 evidence package for '
   11 footer     p3: 'Synthetic training corpus - not for clinical use Page 3'
   12 paragraph  p4: 'Dosage and Administration. The controlled AX-17 evidence package for G'
   13 footer     p4: 'Synthetic training corpus - not for clinical use Page 4'
   14 header     p5: 'Renal Impairment'
   15 paragraph  p5: 'Renal Impairment. The co

In [18]:
# The same function over all 8 files produced output/chunks_cs512.json, which steps 3 and 4 read.
on_disk = json.loads((config.OUTPUT_DIR / f"chunks_cs{config.CHUNK_SIZE}.json").read_text())
print("rows on disk:", len(on_disk), "· this document's rows identical:",
      [r for r in on_disk if r["document_id"] == DOC] == rows)
table([{"document_id": d, "rows": n} for d, n in sorted(Counter(r["document_id"] for r in on_disk).items())])

rows on disk: 250 · this document's rows identical: True


,document_id,rows
0,DE-AX17-HSL-2019-2021,18
1,DE-AX17-LBL-038,29
2,DE-AX17-LBL-042,38
3,HN-204,33
4,HN-311,33
5,HN-402,33
6,SB-AX17-2026-09,37
7,US-AX17-PI-050,29


## Step 3 · the quality gate (`02a_check_parse.py`)

A parser can fail quietly: an empty page, garbled OCR, a split table, a changed word. Before anything is embedded,
`check_file()` compares the parse with the PDF itself: pages in vs out, OCR pages, junk characters, known phrases,
tables intact, sentences cut in the middle, planted instructions, and **faithfulness** (every parsed word of a native
PDF must exist in the PDF's own text layer). A failing file is quarantined and Step 4 skips it.

In [19]:
gate = module("02a_check_parse")
result = gate.check_file(FILE, rows)
for k in ["pages_in", "pages_out", "ocr_pages", "junk_ratio", "phrases_missing", "mid_sentence_endings",
          "injection_hits", "template_only_chunks", "exact_duplicate_chunks", "faithfulness", "quarantine", "problems"]:
    print(f"{k:24} {result.get(k)!r}")
print(f"{'tables intact':24}", {k: v["intact"] for k, v in result.get("tables", {}).items()})

pages_in                 16
pages_out                16
ocr_pages                0
junk_ratio               0.0
phrases_missing          []
mid_sentence_endings     ['DE-AX17-LBL-042-c0013']
injection_hits           []
template_only_chunks     27
exact_duplicate_chunks   0
faithfulness             {'words_not_in_source': {'exposure': 4, 'by': 4, 'function': 4}, 'table_chunks_to_review': []}
quarantine               False
problems                 []
tables intact            {'Table 3': True}


In [20]:
summary = []   # the gate's verdict for every file
for f in config.SOURCE_FILES:
    r = gate.check_file(f, [x for x in on_disk if x["filename"] == f])
    summary.append({"file": f, "pages out/in": f"{r.get('pages_out')}/{r.get('pages_in')}", "ocr_pages": r.get("ocr_pages"),
                    "quarantine": r.get("quarantine"), "problems": r.get("problems"), "notes": r.get("notes")})
table(summary, width=70)

,file,pages out/in,ocr_pages,quarantine,problems,notes
0,DE_AX17_Current_Label_v4.2.pdf,16/16,0,False,[],[]
1,DE_AX17_Superseded_Label_v3.8.pdf,15/15,0,False,[],[]
2,US_AX17_Prescribing_Information_v5.0.pdf,15/15,0,False,[],[table text differs from the PDF in ['US-AX17-...
3,HN-204_Clinical_Study.pdf,17/17,0,False,[],[]
4,HN-311_Clinical_Study.pdf,17/17,0,False,[],[]
5,HN-402_Clinical_Study.pdf,17/17,0,False,[],[]
6,SB_AX17_2026_09_Safety_Bulletin.docx,None/None,0,False,[],"[DOCX: no fixed pages, cite by section]"
7,DE_AX17_Historical_Scanned_Safety_Letters_2019...,18/18,18,False,[],[]


## Step 4 · split tables, tag templates, collapse copies (`03_embed_and_load.py`, steps a–g)

Three problems in this corpus, each with its own rule:
- **Tables mixed with text** → `split_chunk()` gives every table its own piece (title + header + rows). A chart
  description becomes a separate *figure* piece tagged as generated; barcode/QR descriptions are dropped.
- **Template paragraphs** ("…evidence package for Germany included N participants, X reported in Y%…", 90–99 per file
  with different numbers) → `tag_templates()` masks the numbers; a sentence shape found in **≥ 5 different pieces** is
  a template, so the piece is tagged `boilerplate`: kept for audit, never searched. A list in one place (a dose
  schedule written as sentences) repeats its shape inside one piece only, so it stays searchable (D72).
- **Exact copies inside one document** → `collapse_copies()` keeps one and records where the others were.

In [21]:
emb = module("03_embed_and_load")
functions(emb)

,line,function,what it does
0,74,load_inputs,"Chunks, quarantine list, faithfulness flags an..."
1,84,_cell_text,Text inside one table cell (cells hold nested ...
2,94,_header_key,"Header row as a lookup key, e.g. 'adverse even..."
3,99,_raw_parse,The newest cached Layout Parser result for a f...
4,106,table_pages,Table caption or header row (lower case) → pag...
5,133,documents_by_id,Governance fields from the registry table (sou...
6,147,_is_heading,
7,151,_heading_text,
8,155,split_chunk,One Layout Parser chunk → text / table / figur...
9,264,_sentences,


In [22]:
all_chunks, quarantine, review, extraction = emb.load_inputs()
all_chunks = [c for c in all_chunks if c["filename"] not in quarantine]     # a. skip quarantined files
pieces, dropped = [], 0
pages_by_file = {f: emb.table_pages(f) for f in {c["filename"] for c in all_chunks}}
for c in all_chunks:                                                       # b–c. the whole corpus
    split, d = emb.split_chunk(c, pages_by_file[c["filename"]])
    dropped += d
    for p in split:
        p.update(document_id=c["document_id"], filename=c["filename"],
                 needs_review=c["chunk_id"] in review and p["block_type"] == "table")
    pieces.extend(split)
mine = [p for p in pieces if p["document_id"] == DOC]
print(f"{len(all_chunks)} parser rows → {len(pieces)} pieces · {DOC}: {dict(Counter(p['block_type'] for p in mine))}")
print("barcode/QR descriptions dropped (corpus):", dropped)
t3 = next(p for p in mine if p["block_type"] == "table")
print(f"\n{t3['chunk_id']} · {t3['table_id']} · page {t3['page_start']}\n{t3['content']}")

250 parser rows → 262 pieces · DE-AX17-LBL-042: {'text': 36, 'table': 1, 'figure': 1}
barcode/QR descriptions dropped (corpus): 4

DE-AX17-LBL-042-c0012 · Table 3 · page 6
# Table 3. Renal dose modification

| Renal category | eGFR | Starting dose | Maximum | Monitoring |
|-|-|-|-|-|
| Normal | >=90 | 50 mg OD | 100 mg/day | Routine |
| Mild | 60-89 | 50 mg OD | 100 mg/day | Routine |
| Moderate | 30-59 | 25 mg OD | 50 mg/day | Recheck 2-4 weeks |
| Severe | 15-29 | Specialist review | 25 mg/day | Enhanced |


In [23]:
emb.tag_templates(pieces)                        # d. templates
pieces, collapsed = emb.collapse_copies(pieces)   # e. exact copies within a document
per_doc = []
for d in sorted({p["document_id"] for p in pieces}):
    own = [p for p in pieces if p["document_id"] == d]
    per_doc.append({"document_id": d, "pieces": len(own), "searchable": sum(not p["boilerplate"] for p in own),
                    "template (kept, never searched)": sum(p["boilerplate"] for p in own)})
print("exact copies collapsed:", collapsed)
table(per_doc)

exact copies collapsed: 6


,document_id,pieces,searchable,"template (kept, never searched)"
0,DE-AX17-HSL-2019-2021,18,0,18
1,DE-AX17-LBL-038,30,2,28
2,DE-AX17-LBL-042,37,3,34
3,HN-204,33,4,29
4,HN-311,34,4,30
5,HN-402,35,4,31
6,SB-AX17-2026-09,39,3,36
7,US-AX17-PI-050,30,2,28


In [24]:
# g. A document with NOTHING searchable (the scanned letters: only template text) gets a document card,
#    so a question that names it can still find it.
docs = {d["document_id"]: d for d in documents}
scan = "DE-AX17-HSL-2019-2021"
source = [c for c in all_chunks if c["document_id"] == scan]
card = emb.document_card(docs[scan], source, extraction.get(source[0]["filename"], "native"))
card.update(filename=source[0]["filename"], needs_review=False, content_hash=emb._norm_hash(card["content"]))
pieces.append(card)                               # as main() does: the card is loaded with the other pieces
print(short(card["content"], 700))
print(f"\npieces to load: {len(pieces)} · searchable: {sum(not p['boilerplate'] for p in pieces)}")

# DE AX17 Historical Scanned Safety Letters 2019 2021 ⏎  ⏎ Document DE-AX17-HSL-2019-2021 (letter, safety letter). Product AX-17. Jurisdiction: Germany. Period: 2019 to 2021. 18 pages, scanned and read by OCR. Page header: "SCANNED HISTORICAL AX-17 SAFETY LETTER - PAGE 1". Page banner: "ARCHIVE | SUPERSEDED | GERMANY | 2019-2021".

pieces to load: 257 · searchable: 23


## Step 5 · registry fields, embedding text, embedding (`03`, step h)

Every piece gets its governance fields **copied from the registry** (status, countries, version, dates), so search can
filter with a plain `WHERE`. What is embedded is the content plus a short prefix (title | version | countries), so a
chunk read alone still says whose guidance it is. `embed_hash` = SHA-256 of that text + the model name: an unchanged
piece reuses its stored vector, so a re-run makes no embedding call.

In [25]:
d = docs[DOC]
t3.update(title=d["title"], product=d["product"], countries=list(d["countries"]), status=d["status"],
          version=d["version"], document_type=d["document_type"])
text = emb.embedding_text(t3)
tag = f"{config.EMBEDDING_MODEL}@{config.EMBEDDING_DIMENSION}"
print("embedding text:", short(text, 400))
print("embed_hash    :", hashlib.sha256((text + tag).encode()).hexdigest()[:16], "…")

embedding text: DE AX17 Current Label v4.2 | version 4.2 | Germany ⏎  ⏎ # Table 3. Renal dose modification ⏎  ⏎ | Renal category | eGFR | Starting dose | Maximum | Monitoring | ⏎ |-|-|-|-|-| ⏎ | Normal | >=90 | 50 mg OD | 100 mg/day | Routine | ⏎ | Mild | 60-89 | 50 mg OD | 100 mg/day | Routine | ⏎ | Moderate | 30-59 | 25 mg OD | 50 mg/day | Recheck 2-4 weeks | ⏎ | Severe | 15-29 | Specialist review | 25 mg/day | …
embed_hash    : 728076cbded24b93 …


In [26]:
if LIVE:
    from google import genai
    g = genai.Client(vertexai=True, project=config.PROJECT_ID, location=config.EMBEDDING_LOCATION)
    vector = emb._embed_one(g, text, "RETRIEVAL_DOCUMENT")   # documents: RETRIEVAL_DOCUMENT · questions: RETRIEVAL_QUERY
    print(len(vector), "numbers, first five:", [round(v, 4) for v in vector[:5]])
else:
    print("Set LIVE = True to embed this text (one Vertex AI call → 768 numbers).")

Set LIVE = True to embed this text (one Vertex AI call → 768 numbers).


## Step 6 · MERGE into BigQuery `chunks` (shown, not run)

Rows go to a staging table, then one `MERGE` updates changed rows, inserts new ones, and deletes rows that disappeared,
**only for the documents in this run**. Re-running changes nothing, and one new bulletin touches only its own rows.
(The session 07 loader dropped the whole table on every run.)

In [27]:
import inspect
print(inspect.getsource(emb.merge_into_chunks))

def merge_into_chunks(client: bigquery.Client, rows: list[dict], doc_ids: list[str]) -> dict:
    """Staging load + MERGE; delete stale rows only for documents in this run (D23)."""
    staging = table_id(STAGING_TABLE)
    target = table_id(BQ_CHUNKS_TABLE)
    client.create_table(bigquery.Table(target, schema=CHUNKS_SCHEMA), exists_ok=True)
    client.load_table_from_json(rows, staging, job_config=bigquery.LoadJobConfig(
        schema=CHUNKS_SCHEMA, write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)).result()

    updates = ", ".join(f"{c} = S.{c}" for c in COLUMNS if c != "chunk_id")
    job = client.query(f"""
        MERGE `{target}` T
        USING `{staging}` S
        ON T.chunk_id = S.chunk_id
        WHEN MATCHED THEN UPDATE SET {updates}
        WHEN NOT MATCHED THEN INSERT ROW
        WHEN NOT MATCHED BY SOURCE AND T.document_id IN UNNEST(@docs) THEN DELETE
    """, job_config=bigquery.QueryJobConfig(query_parameters=[
        bigquery.ArrayQueryParameter("docs", 

### Step 6b · table facts: one row per table cell (D74)
After the MERGE, every table is also stored cell by cell in BigQuery `table_facts`, read from Layout Parser's
**structured** table (rows × cells, not the Markdown text) and linked to its table piece by the header row and page.
At answer time the **claim check** compares every table value an answer states with its exact cell, so a maximum dose
can never pass as a starting dose (notebook 2, Step 13).

In [28]:
facts = emb.table_facts(pieces)
print(len(facts), "cells from", len({f["chunk_id"] for f in facts}), "tables (every table piece linked:",
      {p["chunk_id"] for p in pieces if p["block_type"] == "table"} == {f["chunk_id"] for f in facts}, ")")
table([f for f in facts if f["table_id"] == "Table 3"], ["table_id", "page", "row_label", "column_label", "value"])

170 cells from 10 tables (every table piece linked: True )


,table_id,page,row_label,column_label,value
0,Table 3,6,Normal,Renal category,Normal
1,Table 3,6,Normal,eGFR,>=90
2,Table 3,6,Normal,Starting dose,50 mg OD
3,Table 3,6,Normal,Maximum,100 mg/day
4,Table 3,6,Normal,Monitoring,Routine
5,Table 3,6,Mild,Renal category,Mild
6,Table 3,6,Mild,eGFR,60-89
7,Table 3,6,Mild,Starting dose,50 mg OD
8,Table 3,6,Mild,Maximum,100 mg/day
9,Table 3,6,Mild,Monitoring,Routine


## Step 7 · verify-searchable probe

"Rows written" is not proof. For each document, code builds a probe phrase from the document's own data and runs a
**real vector search** over the searchable chunks; the run counts as done only when the document comes back.

In [29]:
probes = emb.probe_phrases(pieces, docs)
table([{"document_id": k, "probe phrase": v} for k, v in probes.items()], width=100)

,document_id,probe phrase
0,DE-AX17-HSL-2019-2021,DE AX17 Historical Scanned Safety Letters 2019...
1,DE-AX17-LBL-038,DE AX17 Superseded Label v3.8 Historical renal...
2,DE-AX17-LBL-042,DE AX17 Current Label v4.2 Table 3. Renal dose...
3,HN-204,HN-204 Clinical Study Table 14. Treatment-emer...
4,HN-311,HN-311 Clinical Study Table 14. Treatment-emer...
5,HN-402,HN-402 Clinical Study Table 14. Treatment-emer...
6,SB-AX17-2026-09,SB AX17 2026 09 Safety Bulletin HelixNova Safe...
7,US-AX17-PI-050,US AX17 Prescribing Information v5.0 US renal ...


In [30]:
if LIVE:
    import time
    from google import genai
    from google.cloud import bigquery
    bq = bigquery.Client(project=config.PROJECT_ID)
    g = genai.Client(vertexai=True, project=config.PROJECT_ID, location=config.EMBEDDING_LOCATION)
    found = emb.verify_searchable(bq, g, {DOC: probes[DOC]}, Health("notebook"), time.time())
    print("found by search:", found, "of 1")
else:
    print("Set LIVE = True to run the probe for", DOC, "(one embedding + one BigQuery VECTOR_SEARCH, read-only).")

Set LIVE = True to run the probe for DE-AX17-LBL-042 (one embedding + one BigQuery VECTOR_SEARCH, read-only).


## Step 8 · the vector index (`04_create_index.py`)

Below 5,000 rows BigQuery refuses an IVF index and searches exactly (brute force), which is fast at this size. At
production scale (millions of chunks) the index is built with the filter columns **stored in it**, so the filters run
before the nearest-neighbour search instead of after it.

In [31]:
idx = module("04_create_index")
print("minimum rows for an IVF index:", idx.MIN_ROWS_IVF)
print("this corpus:", len(pieces), "rows →", "index skipped, exact search" if len(pieces) < idx.MIN_ROWS_IVF else "index built")
for name, definition in idx.DEFINITIONS:
    print(f"- {name}: {definition}")

minimum rows for an IVF index: 5000
this corpus: 257 rows → index skipped, exact search
- stored filters + lexical column: STORING(content, document_id, product, countries, status, effective_from, valid_to, boilerplate, embedding_model, document_type) OPTIONS(index_type = 'IVF', distance_type = 'COSINE', lexical_search_columns = ['content'])
- stored filters: STORING(document_id, product, countries, status, effective_from, valid_to, boilerplate, embedding_model, document_type) OPTIONS(index_type = 'IVF', distance_type = 'COSINE')
- stored filters without countries: STORING(document_id, product, status, effective_from, valid_to, boilerplate, embedding_model, document_type) OPTIONS(index_type = 'IVF', distance_type = 'COSINE')
- plain: OPTIONS(index_type = 'IVF', distance_type = 'COSINE')


In [32]:
if LIVE:   # what BigQuery `chunks` holds per document (read-only)
    from google.cloud import bigquery
    bq = bigquery.Client(project=config.PROJECT_ID)
    rows = [dict(r.items()) for r in bq.query(f"""
        SELECT document_id, ANY_VALUE(status) AS status, COUNT(*) AS chunks, COUNTIF(NOT boilerplate) AS searchable,
               COUNTIF(block_type = 'table') AS tables, ANY_VALUE(embedding_model) AS model
        FROM `{config.table_id('chunks')}` GROUP BY document_id ORDER BY document_id""").result()]
    display(table(rows))
else:
    print("Set LIVE = True to count what BigQuery `chunks` holds per document.")

Set LIVE = True to count what BigQuery `chunks` holds per document.


## Recap: what ingestion guarantees

| Step | Function(s) | Produces | Why it matters at question time |
|---|---|---|---|
| 1 | `read_documents`, `_derive_valid_to`, `check_quality` | `documents`, `adverse_events`, two views | status / country / dates become **filters before search**; counts come from SQL |
| 2 | `parse_file`, `_index_blocks`, `_chunk_rows` | chunk rows with pages and block kinds | every answer can cite a **page**; model-written text kept apart |
| 3 | `check_file` | pass / quarantine per file | a bad parse never reaches the index |
| 4 | `split_chunk`, `tag_templates`, `collapse_copies`, `document_card` | 257 pieces, 23 searchable | tables whole, template decoys never searched |
| 5 | `embedding_text`, `embed_hash`, `_embed_one` | 768-number vectors | meaning search; re-runs reuse vectors |
| 6 | `merge_into_chunks`, `table_facts` | BigQuery `chunks` + `table_facts` (one row per table cell) | safe re-runs; table values checked cell by cell |
| 7 | `probe_phrases`, `verify_searchable` | pass / fail per document | "loaded" means **found by search** |
| 8 | `04_create_index.main` | index only at scale | exact search now, indexed search later |

Next: `02_query_walkthrough.ipynb`, one question through these chunks.